In [1]:
###
###     * * * IMPORTANTE * * *
###   Definir Carpeta de trabajo con archivos de datos
import os
os.chdir(r"C:\Users\ariel\OneDrive\Escritorio\Otoño2026\Redes\Archivos Berlin y Anaheim")

# Importar Librerías
from collections import defaultdict
import math
import heapq
import time
from collections import deque

# Parámetros Resolución
# C1 = 2
# C2 = 7
# -> Nodo inicio Berlin : 322
# -> Nodo inicio Anaheim: 127



In [2]:
##
## Lectura del archivo Berlín
##

#   Creación de lista de datos de Berlin
grafo_berlin = defaultdict(list)

with open("Berlin/berlin_net.txt") as f:

    for linea in f:

        partes = linea.replace(";", "").split()

        if len(partes) >= 5:

            try:

                origen = int(partes[0])
                destino = int(partes[1])
                tiempo = float(partes[4])

                grafo_berlin[origen].append((destino, tiempo))

            except:
                pass

#   Ejemplo de Prueba
print("Nodos con salida:", len(grafo_berlin))
print("Arcos desde nodo 1:", grafo_berlin[1][:5])

Nodos con salida: 966
Arcos desde nodo 1: [(817, 0.0), (818, 0.0), (821, 0.0), (822, 0.0)]


In [3]:
##
## Lectura del archivo Berlín
##

#   Creación de lista de datos de Berlin
grafo_anaheim = defaultdict(list)

with open("Anaheim\\AnaheimNet.tntp") as f:

    for linea in f:

        partes = linea.replace(";", "").split()

        try:

            origen = int(partes[0])
            destino = int(partes[1])
            tiempo = float(partes[4])

            grafo_anaheim[origen].append((destino, tiempo))

        except:
            pass

#   Ejemplo de Prueba
print("Nodos con salida:", len(grafo_anaheim))
print("Arcos desde nodo 1:", grafo_anaheim[1][:5])

Nodos con salida: 416
Arcos desde nodo 1: [(117, 1.090458488)]


In [4]:
##       ALGORITMO       ##########################################################################################################################
##          DE           ##########################################################################################################################
##       DIJKSTRA        ##########################################################################################################################



def dijkstra_heap(grafo, origen):
    dist = {}
    pred = {}
    visitado = set()
    heap = []

    #   Inicialización del algoritmo
    dist[origen] = 0
    pred[origen] = None
    heapq.heappush(heap, (0, origen))

    iteraciones = 0
    extracciones_heap = 0
    relajaciones = 0

    while heap:
        distancia_actual, nodo_actual = heapq.heappop(heap)
        extracciones_heap += 1

        if nodo_actual in visitado:
            continue

        visitado.add(nodo_actual)
        iteraciones += 1

        for vecino, costo in grafo[nodo_actual]:
            nueva_distancia = distancia_actual + costo
            relajaciones += 1

            if vecino not in dist or nueva_distancia < dist[vecino]:
                dist[vecino] = nueva_distancia
                pred[vecino] = nodo_actual
                heapq.heappush(heap, (nueva_distancia, vecino))
    
    #   Indicadores
    metricas = {
        "iteraciones": iteraciones,
        "extracciones_heap": extracciones_heap,
        "relajaciones": relajaciones,
        "nodos_visitados": len(visitado)
    }

    return dist, pred, metricas

In [5]:
##       ALGORITMO       ##########################################################################################################################
##          DE           ##########################################################################################################################
##      D'ESOPO PAPE     ##########################################################################################################################
def desopo_pape(grafo, origen):
    dist = {origen: 0}
    pred = {origen: None}
    
    cola = deque([origen])
    estado = defaultdict(int)  
    # 0 = nunca visto
    # 1 = en cola
    # 2 = procesado
    
    estado[origen] = 1

    iteraciones = 0
    relajaciones = 0
    reinserciones_frente = 0
    inserciones_final = 0

    while cola:
        i = cola.popleft()
        estado[i] = 2
        iteraciones += 1

        for j, costo in grafo[i]:
            relajaciones += 1
            nueva_dist = dist[i] + costo

            if j not in dist or nueva_dist < dist[j]:
                dist[j] = nueva_dist
                pred[j] = i

                if estado[j] == 0:
                    cola.append(j)
                    estado[j] = 1
                    inserciones_final += 1

                elif estado[j] == 2:
                    cola.appendleft(j)
                    estado[j] = 1
                    reinserciones_frente += 1

    metricas = {
        "iteraciones": iteraciones,
        "relajaciones": relajaciones,
        "inserciones_final": inserciones_final,
        "reinserciones_frente": reinserciones_frente,
        "nodos_alcanzados": len(dist)
    }

    return dist, pred, metricas

In [6]:
##
##   Función de EXtracción de Resultados
##


def reconstruir_ruta(pred, origen, destino):
    if destino not in pred and destino != origen:
        return None

    ruta = []
    nodo = destino

    while nodo is not None:
        ruta.append(nodo)
        nodo = pred.get(nodo)

    ruta.reverse()

    if ruta[0] != origen:
        return None

    return ruta

In [8]:
##
##   BERLIN
##

#   Nodo Origen
origen_berlin = 322
#   Nodos destino
destinos_berlin = [751, 633, 477, 122]

#   DIJKSTRA
#   Inicio Timer
inicio = time.perf_counter()
#   Algoritmo
dist_dijkstra, pred_dijkstra , metricas_dijkstra = dijkstra_heap(grafo_berlin , origen_berlin)
#   Fin Timer
fin = time.perf_counter()
#   Tiempo de computación
tiempo_dijkstra = fin - inicio


#   D'ESOPO PAPE
#   Inicio Timer
inicio = time.perf_counter()
#   Algoritmo
dist_desopo , pred_desopo , metricas_desopo = desopo_pape(grafo_berlin , origen_berlin)
#   Fin Timer
fin = time.perf_counter()
#   Tiempo de computación
tiempo_desopo = fin - inicio

#   RESULTADOS
#   Extracción rutas y costos por destino
print("*  *  *  *  *  *  *  *  *  *")
print("*  *  *  RESULTADOS  *  *  *")
print("*  *  *  *  *  *  *  *  *  *")
print()
for destino in destinos_berlin:
    ruta_dijkstra = reconstruir_ruta(pred_dijkstra , origen_berlin , destino)
    ruta_desopo_p = reconstruir_ruta(pred_desopo , origen_berlin , destino)
    print("Destino:", destino)
    print("Costo Dijkstra:", dist_dijkstra.get(destino, "No alcanzable"))
    print("Costo D'Esopo-Pape:", dist_desopo[destino])
    print("Diferencia Costos:", abs(dist_dijkstra[destino] - dist_desopo[destino]))
    print("Ruta Dijkstra :", ruta_dijkstra)
    print("Ruta D'esopo-Pape :", ruta_desopo_p)
    print()
    print(" -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - -")
    print()

#   Metricas de rendimiento de cada algoritmo
print()
print("*  *  *  *  *  *  *   *  *  *  *  *  *  *")
print("*  *  *  Métricas de rendimiento  *  *  *")
print("*  *  *  *  *  *  *   *  *  *  *  *  *  *")
print()
print("Tiempo Dijkstra:", tiempo_dijkstra)
print(metricas_dijkstra)
print()
print("Tiempo D'Esopo-Pape:", tiempo_desopo)
print(metricas_desopo)
print()


*  *  *  *  *  *  *  *  *  *
*  *  *  RESULTADOS  *  *  *
*  *  *  *  *  *  *  *  *  *

Destino: 751
Costo Dijkstra: 26.666666
Costo D'Esopo-Pape: 26.666666
Diferencia Costos: 0.0
Ruta Dijkstra : [322, 81, 374, 748, 972, 754, 2, 751]
Ruta D'esopo-Pape : [322, 81, 374, 748, 972, 754, 2, 751]

 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - -

Destino: 633
Costo Dijkstra: 129.33333299999998
Costo D'Esopo-Pape: 129.33333299999998
Diferencia Costos: 0.0
Ruta Dijkstra : [322, 380, 790, 793, 707, 685, 223, 7, 241, 243, 244, 962, 645, 670, 673, 648, 15, 660, 662, 661, 580, 579, 82, 571, 577, 559, 560, 637, 34, 633]
Ruta D'esopo-Pape : [322, 380, 790, 793, 707, 685, 223, 7, 241, 243, 244, 962, 645, 670, 673, 648, 15, 660, 662, 661, 580, 579, 82, 571, 577, 559, 560, 637, 34, 633]

 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - -

Destino: 4

In [16]:
##
##   ANAHEIM
##

#   Nodo Origen
origen_anaheim = 127
#   Nodos Destino
destinos_anaheim = [380, 270]


#   DIJKSTRA
#   Inicio Timer
inicio = time.perf_counter()
#   Algoritmo
dist_dijkstra_a , pred_dijkstra_a , met_dijkstra_a = dijkstra_heap(grafo_anaheim , origen_anaheim)
#   Fin Timer
fin = time.perf_counter()
#   Tiempo de computación
tiempo_dijkstra_a = fin - inicio


#   D'ESOPO PAPE
#   Inicio Timer
inicio = time.perf_counter()
#   Algoritmo
dist_desopo_a , pred_desopo_a , met_desopo_a = desopo_pape(grafo_anaheim , origen_anaheim)
#   Fin Timer
fin = time.perf_counter()
#   Tiempo de computación
tiempo_desopo_a = fin - inicio



#   RESULTADOS
#   Extracción rutas y costos por destino
#   RESULTADOS
#   Extracción rutas y costos por destino
print("*  *  *  *  *  *  *  *  *  *")
print("*  *  *  RESULTADOS  *  *  *")
print("*  *  *  *  *  *  *  *  *  *")
print()
for destino in destinos_anaheim:
    ruta_dijkstra_a = reconstruir_ruta(pred_dijkstra_a , origen_anaheim , destino)
    ruta_desopo_p_a = reconstruir_ruta(pred_desopo_a , origen_anaheim , destino)
    print("Destino:", destino)
    print("Costo Dijkstra:", dist_dijkstra_a.get(destino, "No alcanzable"))
    print("Costo D'Esopo-Pape:", dist_desopo_a[destino])
    print("Diferencia Costos:", abs(dist_dijkstra_a[destino] - dist_desopo_a[destino]))
    print("Ruta Dijkstra :", ruta_dijkstra_a)
    print("Ruta D'esopo-Pape :", ruta_desopo_p_a)
    print()
    print(" -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - -")
    print()

#   Metricas de rendimiento de cada algoritmo
print()
print("*  *  *  *  *  *  *   *  *  *  *  *  *  *")
print("*  *  *  Métricas de rendimiento  *  *  *")
print("*  *  *  *  *  *  *   *  *  *  *  *  *  *")
print()
print("Tiempo Dijkstra:", tiempo_dijkstra_a)
print(met_dijkstra_a)
print()
print("Tiempo D'Esopo-Pape:", tiempo_desopo_a)
print(met_desopo_a)
print()



*  *  *  *  *  *  *  *  *  *
*  *  *  RESULTADOS  *  *  *
*  *  *  *  *  *  *  *  *  *

Destino: 380
Costo Dijkstra: 5.268071044
Costo D'Esopo-Pape: 5.268071044
Diferencia Costos: 0.0
Ruta Dijkstra : [127, 126, 125, 124, 123, 382, 381, 380]
Ruta D'esopo-Pape : [127, 126, 125, 124, 123, 382, 381, 380]

 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - -

Destino: 270
Costo Dijkstra: 11.483509971999998
Costo D'Esopo-Pape: 11.483509971999998
Diferencia Costos: 0.0
Ruta Dijkstra : [127, 350, 349, 156, 155, 154, 153, 152, 151, 150, 149, 148, 147, 146, 145, 144, 264, 265, 266, 24, 267, 268, 25, 269, 270]
Ruta D'esopo-Pape : [127, 350, 349, 156, 155, 154, 153, 152, 151, 150, 149, 148, 147, 146, 145, 144, 264, 265, 266, 24, 267, 268, 25, 269, 270]

 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - -


*  *  *  *  *  *  *   *  *  *  *  *  *  *
*